In [23]:
import sys
!{sys.executable} -m pip install pytorch-msssim


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# 1. CNN Encoder
# ============================================================================
class CNNEncoder(nn.Module):
    def __init__(self, in_channels=1, out_channels=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, out_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.encoder(x)  # [B, C, H/16, W/16]

# ============================================================================
# 2. Temporal Transformer
# ============================================================================
class TemporalTransformer(nn.Module):
    def __init__(self, dim=64, num_heads=4, time_embed_dim=32):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(time_embed_dim, dim)
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm_cross = nn.LayerNorm(dim)
        self.self_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.cross_attn_f0 = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.cross_attn_f1 = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        self.norm_ffn = nn.LayerNorm(dim)

    def forward(self, f0, f1, timestep):
        B, C, H, W = f0.shape
        if timestep.dim() == 4:
            timestep = timestep.view(B, 1)
        elif timestep.dim() == 1:
            timestep = timestep.unsqueeze(1)
        time_emb = self.time_embed(timestep).unsqueeze(1)  # [B,1,C]
        f0_seq = f0.flatten(2).transpose(1, 2)   # [B, N, C]
        f1_seq = f1.flatten(2).transpose(1, 2)
        f0_seq = f0_seq + time_emb
        f1_seq = f1_seq + time_emb
        # Self-Attn
        attn_f0 = self.self_attn(self.norm1(f0_seq), self.norm1(f0_seq), self.norm1(f0_seq))[0]
        f0_seq = f0_seq + attn_f0
        attn_f1 = self.self_attn(self.norm1(f1_seq), self.norm1(f1_seq), self.norm1(f1_seq))[0]
        f1_seq = f1_seq + attn_f1
        # Cross-Attn
        f0_cross = self.cross_attn_f0(
            query=self.norm_cross(f0_seq),
            key=self.norm_cross(f1_seq),
            value=self.norm_cross(f1_seq)
        )[0]
        f0_seq = f0_seq + f0_cross
        f1_cross = self.cross_attn_f1(
            query=self.norm_cross(f1_seq),
            key=self.norm_cross(f0_seq),
            value=self.norm_cross(f0_seq)
        )[0]
        f1_seq = f1_seq + f1_cross
        # FFN
        f0_seq = f0_seq + self.ffn(self.norm_ffn(f0_seq))
        f1_seq = f1_seq + self.ffn(self.norm_ffn(f1_seq))
        f0_out = f0_seq.transpose(1, 2).reshape(B, C, H, W)
        f1_out = f1_seq.transpose(1, 2).reshape(B, C, H, W)
        return f0_out, f1_out

# ============================================================================
# 3. Flow Network
# ============================================================================
class RIFEFlowNet(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.conv_in = nn.Conv2d(2 * feature_dim, 128, kernel_size=3, padding=1)
        self.encoder = nn.Sequential(
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 3, stride=2, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 2, 3, padding=1)
        )

    def forward(self, f0, f1):
        x = torch.cat([f0, f1], dim=1)
        x = self.conv_in(x)
        x = self.encoder(x)
        flow = self.decoder(x)
        return flow  # [B,2, H/16, W/16]

# ============================================================================
# 4. Warping & Fusion
# ============================================================================
class WarpingModule(nn.Module):
    def forward(self, frame, flow):
        B, _, H, W = frame.shape
        grid_y, grid_x = torch.meshgrid(
            torch.arange(H, device=frame.device),
            torch.arange(W, device=frame.device),
            indexing='ij'
        )
        grid = torch.stack([grid_x, grid_y], dim=0).float()
        grid = grid.unsqueeze(0).expand(B, -1, -1, -1).clone()
        grid = grid + flow
        grid[:, 0] = 2.0 * grid[:, 0] / (W - 1) - 1.0
        grid[:, 1] = 2.0 * grid[:, 1] / (H - 1) - 1.0
        grid = grid.permute(0, 2, 3, 1)
        return F.grid_sample(frame, grid, mode='bilinear', align_corners=True)

class FusionModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Conv2d(6, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, warped0, warped1, frame0, frame2, flow):
        x = torch.cat([warped0, warped1, frame0, frame2, flow], dim=1)
        return self.fusion(x)

# ============================================================================
# 5. Complete Model
# ============================================================================
class CNN_Attention_RIFE_Temporal(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.encoder = CNNEncoder(in_channels=1, out_channels=feature_dim)
        self.transformer = TemporalTransformer(dim=feature_dim)
        self.flow_net = RIFEFlowNet(feature_dim=feature_dim)
        self.warper = WarpingModule()
        self.fusion = FusionModule()

    def forward(self, frame0, frame2, timestep):
        B, _, H, W = frame0.shape
        timestep_4d = timestep.view(B, 1, 1, 1)

        f0 = self.encoder(frame0)   # [B, C, H/16, W/16]
        f2 = self.encoder(frame2)
        f0, f2 = self.transformer(f0, f2, timestep)

        flow = self.flow_net(f0, f2)   # [B, 2, H/16, W/16]

        flow_up = F.interpolate(flow, size=(H, W), mode='bilinear', align_corners=True)
        scale_factor = H / flow.shape[2]   # = 16
        flow_up = flow_up * scale_factor

        warped0 = self.warper(frame0, flow_up * timestep_4d)
        warped2 = self.warper(frame2, -flow_up * (1 - timestep_4d))

        flow_normalised = flow_up / (scale_factor + 1e-6)
        output = self.fusion(warped0, warped2, frame0, frame2, flow_normalised)
        return output

# ============================================================================
# 6. Utilities (padding/cropping)
# ============================================================================
def pad_to_multiple_of_16(x: torch.Tensor):
    _, _, H, W = x.shape
    pad_h = (16 - H % 16) % 16
    pad_w = (16 - W % 16) % 16
    if pad_h > 0 or pad_w > 0:
        x = F.pad(x, (0, pad_w, 0, pad_h), mode='reflect')
    return x, H, W

def crop_to_original(x: torch.Tensor, H: int, W: int) -> torch.Tensor:
    return x[:, :, :H, :W]

# ============================================================================
# 7. Load your trained model
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate the architecture (must match training: feature_dim=64)
model = CNN_Attention_RIFE_Temporal(feature_dim=64)

# Load checkpoint with robust path resolution
model_path = '../backend/best_model_512.pth'
if not os.path.exists(model_path):
    model_path = 'best_model_512.pth'  # fallback
    
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

# Extract normalisation stats
global_min = float(checkpoint['global_min'])
global_max = float(checkpoint['global_max'])

print(f"✅ Model loaded from epoch {checkpoint['epoch']}")
print(f"   Val SSIM: {checkpoint['val_ssim']:.4f}")
print(f"   Normalisation range: [{global_min:.2f}, {global_max:.2f}]")

In [26]:
# ============================================================================
# INFERENCE PIPELINE: Load two .nc files, interpolate, save result
# ============================================================================
import numpy as np
import torch
import xarray as xr
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# -------------------------------------------------------------
# 1. Define helper functions for patch extraction & merging
# -------------------------------------------------------------
def extract_patches_overlap(data, patch_size=512, overlap=64):
    """
    Extract overlapping patches from a 2D array.
    Returns: list of (y_top, x_left, patch_array)
    """
    H, W = data.shape
    step = patch_size - overlap
    patches = []
    
    y_starts = list(range(0, H - patch_size + 1, step))
    x_starts = list(range(0, W - patch_size + 1, step))
    
    # Ensure we cover the edges
    if y_starts[-1] + patch_size < H:
        y_starts.append(H - patch_size)
    if x_starts[-1] + patch_size < W:
        x_starts.append(W - patch_size)
    
    for y in y_starts:
        for x in x_starts:
            patch = data[y:y+patch_size, x:x+patch_size]
            patches.append((y, x, patch))
    
    return patches

def merge_patches(patches_with_coords, output_shape, patch_size=512, overlap=64):
    """
    Stitch patches back together with a Hanning window to reduce seams.
    """
    H, W = output_shape
    accum = np.zeros(output_shape, dtype=np.float64)
    weight_map = np.zeros(output_shape, dtype=np.float64)
    
    # Create a 2D Hanning window
    win_1d = np.hanning(patch_size)
    win_2d = np.outer(win_1d, win_1d)
    
    for y, x, patch in patches_with_coords:
        accum[y:y+patch_size, x:x+patch_size] += patch * win_2d
        weight_map[y:y+patch_size, x:x+patch_size] += win_2d
    
    weight_map = np.maximum(weight_map, 1e-8)
    output = accum / weight_map
    return output.astype(np.float32)

# -------------------------------------------------------------
# 2. Define interpolation function for single patches (for batching)
# -------------------------------------------------------------
def interpolate_patch_batch(model, patches0, patches2, timestep=0.5, 
                            global_min=None, global_max=None, device='cuda'):
    """
    Process a batch of patches efficiently.
    """
    rng = global_max - global_min + 1e-8
    B = len(patches0)
    
    # Normalize patches
    batch0 = []
    batch2 = []
    for p0, p2 in zip(patches0, patches2):
        p0_norm = np.clip((p0 - global_min) / rng, 0.0, 1.0)
        p2_norm = np.clip((p2 - global_min) / rng, 0.0, 1.0)
        batch0.append(torch.from_numpy(p0_norm).unsqueeze(0).unsqueeze(0).float())
        batch2.append(torch.from_numpy(p2_norm).unsqueeze(0).unsqueeze(0).float())
    
    batch0 = torch.cat(batch0, dim=0).to(device)
    batch2 = torch.cat(batch2, dim=0).to(device)
    timestep_tensor = torch.full((B, 1), timestep, device=device)
    
    with torch.no_grad():
        preds = model(batch0, batch2, timestep_tensor)
    
    # Denormalize and convert back to numpy
    results = []
    for i in range(B):
        pred_norm = preds[i].squeeze().cpu().numpy()
        pred_denorm = (pred_norm * rng) + global_min
        results.append(pred_denorm)
    
    return results

In [ ]:
# ============================================================================
# INFERENCE PIPELINE with Dynamic Timestamp & Ground Truth
# ============================================================================
import os
import re
from datetime import datetime
import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_msssim import ssim
import torch.nn.functional as F
import warnings
warnings.filterwarnings("ignore")

# -------------------------------------------------------------
# 1. Helper functions (patch extraction & merging)
# -------------------------------------------------------------
def extract_patches_overlap(data, patch_size=512, overlap=64):
    H, W = data.shape
    step = patch_size - overlap
    patches = []
    y_starts = list(range(0, H - patch_size + 1, step))
    x_starts = list(range(0, W - patch_size + 1, step))
    if y_starts[-1] + patch_size < H:
        y_starts.append(H - patch_size)
    if x_starts[-1] + patch_size < W:
        x_starts.append(W - patch_size)
    for y in y_starts:
        for x in x_starts:
            patch = data[y:y+patch_size, x:x+patch_size]
            patches.append((y, x, patch))
    return patches

def merge_patches(patches_with_coords, output_shape, patch_size=512, overlap=64):
    H, W = output_shape
    accum = np.zeros(output_shape, dtype=np.float64)
    weight_map = np.zeros(output_shape, dtype=np.float64)
    win_1d = np.hanning(patch_size)
    win_2d = np.outer(win_1d, win_1d)
    for y, x, patch in patches_with_coords:
        accum[y:y+patch_size, x:x+patch_size] += patch * win_2d
        weight_map[y:y+patch_size, x:x+patch_size] += win_2d
    weight_map = np.maximum(weight_map, 1e-8)
    return (accum / weight_map).astype(np.float32)

# -------------------------------------------------------------
# 2. Batch interpolation function
# -------------------------------------------------------------
def interpolate_patch_batch(model, patches0, patches2, timestep=0.5,
                            global_min=None, global_max=None, device='cuda'):
    rng = global_max - global_min + 1e-8
    B = len(patches0)
    batch0, batch2 = [], []
    for p0, p2 in zip(patches0, patches2):
        p0_norm = np.clip((p0 - global_min) / rng, 0.0, 1.0)
        p2_norm = np.clip((p2 - global_min) / rng, 0.0, 1.0)
        batch0.append(torch.from_numpy(p0_norm).unsqueeze(0).unsqueeze(0).float())
        batch2.append(torch.from_numpy(p2_norm).unsqueeze(0).unsqueeze(0).float())
    batch0 = torch.cat(batch0, dim=0).to(device)
    batch2 = torch.cat(batch2, dim=0).to(device)
    ts = torch.full((B, 1), timestep, device=device)
    with torch.no_grad():
        preds = model(batch0, batch2, ts)
    results = []
    for i in range(B):
        pred_norm = preds[i].squeeze().cpu().numpy()
        results.append((pred_norm * rng) + global_min)
    return results

# -------------------------------------------------------------
# 3. Parse timestamp from filename
# -------------------------------------------------------------
def parse_timestamp_from_filename(filename):
    """
    Extract timestamp from ABI filename.
    Example: 'OR_ABI-L2-CMIPF-M6C13_G19_s20242970100193_e...'
    Returns: datetime object
    """
    base = os.path.basename(filename)
    match = re.search(r"_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})", base)
    if match is None:
        raise ValueError(f"Cannot parse timestamp from: {base}")
    year = int(match.group(1))
    doy = int(match.group(2))
    hour = int(match.group(3))
    minute = int(match.group(4))
    second = int(match.group(5))
    return datetime.strptime(
        f"{year} {doy} {hour}:{minute}:{second}",
        "%Y %j %H:%M:%S"
    )

# -------------------------------------------------------------
# 4. Main inference
# -------------------------------------------------------------

# ===== USER INPUT: CHANGE THESE =====
nc_file_0 = "../OR_ABI-L2-CMIPF-M6C13_G19_s20252960740213_e20252960749534_c20252960749583.nc"
nc_file_2 = "../OR_ABI-L2-CMIPF-M6C13_G19_s20252960800213_e20252960809533_c20252960809581.nc"
nc_file_1 = "../OR_ABI-L2-CMIPF-M6C13_G19_s20252960750213_e20252960759534_c20252960759594.nc"   # Ground truth (t=1)
variable_name = "CMI"
PATCH_SIZE = 512
OVERLAP = 64
BATCH_SIZE = 8

# ===== DYNAMIC TIMESTEP: Calculate from filenames =====
print("⏱️  Parsing timestamps from filenames...")
t0_dt = parse_timestamp_from_filename(nc_file_0)
t1_dt = parse_timestamp_from_filename(nc_file_1) if nc_file_1 is not None else None
t2_dt = parse_timestamp_from_filename(nc_file_2)

if t0_dt is not None and t1_dt is not None and t2_dt is not None:
    total_sec = (t2_dt - t0_dt).total_seconds()
    current_sec = (t1_dt - t0_dt).total_seconds()
    if total_sec > 0:
        timestep = np.clip(current_sec / total_sec, 0.0, 1.0)
    else:
        timestep = 0.5
else:
    timestep = 0.5  # fallback

print(f"   t0: {t0_dt.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   t1: {t1_dt.strftime('%Y-%m-%d %H:%M:%S') if t1_dt else 'N/A'}")
print(f"   t2: {t2_dt.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Calculated timestep: {timestep:.4f}")

# ===== Load .nc files =====
print(f"\n📂 Loading {nc_file_0} ...")
ds0 = xr.open_dataset(nc_file_0)
data0 = ds0[variable_name].values.astype(np.float32)

print(f"📂 Loading {nc_file_2} ...")
ds2 = xr.open_dataset(nc_file_2)
data2 = ds2[variable_name].values.astype(np.float32)

if nc_file_1 is not None:
    print(f"📂 Loading ground truth {nc_file_1} ...")
    ds1 = xr.open_dataset(nc_file_1)
    data_gt = ds1[variable_name].values.astype(np.float32)
else:
    data_gt = None

print(f"   Arrays: {data0.shape} and {data2.shape}")
if data_gt is not None:
    print(f"   Ground truth shape: {data_gt.shape}")

# fill missing values
data0 = np.nan_to_num(data0, nan=global_min)
data2 = np.nan_to_num(data2, nan=global_min)
if data_gt is not None:
    data_gt = np.nan_to_num(data_gt, nan=global_min)

# ===== Extract patches =====
print(f"\n🧩 Extracting patches (size={PATCH_SIZE}, overlap={OVERLAP}) ...")
patches0 = extract_patches_overlap(data0, PATCH_SIZE, OVERLAP)
patches2 = extract_patches_overlap(data2, PATCH_SIZE, OVERLAP)
print(f"   Total patches: {len(patches0)}")

# ===== Run inference =====
output_patches = []
model.eval()
print(f"🧠 Running inference (batch size {BATCH_SIZE}) ...")
for i in tqdm(range(0, len(patches0), BATCH_SIZE)):
    end = min(i + BATCH_SIZE, len(patches0))
    batch_p0 = [p for _, _, p in patches0[i:end]]
    batch_p2 = [p for _, _, p in patches2[i:end]]
    coords = [(y, x) for y, x, _ in patches0[i:end]]
    preds = interpolate_patch_batch(
        model, batch_p0, batch_p2, timestep,   # <-- using dynamic timestep here
        global_min, global_max, device
    )
    for (y, x), pred in zip(coords, preds):
        output_patches.append((y, x, pred))

# ===== Merge =====
print("🔄 Merging patches ...")
final_img = merge_patches(output_patches, data0.shape, PATCH_SIZE, OVERLAP)

# ===== Save results =====
out_nc = "interpolated_frame.nc"
ds_out = xr.Dataset(
    {variable_name: (['y', 'x'], final_img)},
    coords={'y': np.arange(final_img.shape[0]),
            'x': np.arange(final_img.shape[1])}
)
ds_out.to_netcdf(out_nc)
np.save("interpolated_frame.npy", final_img)
print(f"💾 Saved interpolated image to {out_nc} and .npy")

# ===== Compute metrics if ground truth exists =====
ssim_val = None
psnr_val = None
if data_gt is not None:
    H, W = final_img.shape
    gt_cropped = data_gt[:H, :W]
    rng = global_max - global_min + 1e-8
    interp_norm = np.clip((final_img - global_min) / rng, 0.0, 1.0)
    gt_norm = np.clip((gt_cropped - global_min) / rng, 0.0, 1.0)
    interp_t = torch.from_numpy(interp_norm).unsqueeze(0).unsqueeze(0).float().to(device)
    gt_t = torch.from_numpy(gt_norm).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        ssim_val = ssim(interp_t, gt_t, data_range=1.0, size_average=True).item()
        mse_val = F.mse_loss(interp_t, gt_t).item()
        psnr_val = 10 * np.log10(1.0 / (mse_val + 1e-10))
    print(f"\n📊 Comparison with Ground Truth (t=1):")
    print(f"   SSIM = {ssim_val:.4f}")
    print(f"   PSNR = {psnr_val:.2f} dB")
else:
    print("\n⚠️ No ground truth provided – skipping metrics.")

# ===== Colourised 4‑panel comparison =====
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
vmin = np.percentile(final_img, 2)
vmax = np.percentile(final_img, 98)

im0 = axes[0].imshow(data0, cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title("Frame 0 (t=0)")
axes[0].axis('off')

im1 = axes[1].imshow(final_img, cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title(f"Interpolated (t={timestep:.3f})")
axes[1].axis('off')

if data_gt is not None:
    im2 = axes[2].imshow(gt_cropped, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[2].set_title(f"Ground Truth (t=1)\nSSIM={ssim_val:.3f}, PSNR={psnr_val:.1f}dB")
    axes[2].axis('off')
    idx_last = 3
else:
    idx_last = 2

im3 = axes[idx_last].imshow(data2, cmap='viridis', vmin=vmin, vmax=vmax)
axes[idx_last].set_title("Frame 2 (t=2)")
axes[idx_last].axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im0, cax=cbar_ax, label=variable_name)
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig("interpolation_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Colour comparison plot saved as interpolation_comparison.png")